# Prompt Engineering 101 - Part III. HOMEWORK
## Pattern-Based Engineering

In [ ]:
# @title 🛠️ Step 1: Laboratory Setup (Gemini API)
# We are connecting to Google's Gemini models.

# 1. Install the Google AI SDK
!pip install -q -U google-genai


from google.colab import userdata
import google.genai as genai

from IPython.display import display, Markdown


# 2. Configure the API Key
# Go to https://aistudio.google.com/app/apikey to get a key.
# It is free and takes 1 click.

# 3. Use Colab Secrets (Best Practice)
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# 4. Create Wrapper Class for querying
class GeminiModel:
    def __init__(self, api_key, preferred_model='gemini-2.5-flash'):
        self.client = genai.Client(api_key=api_key)
        
        # Priority list of models and their availability status
        # True = Available, False = Exhausted (429)
        self.models = {
            'gemini-2.5-flash': True,
            'gemini-2.5-flash-lite': True,
            'gemini-3-flash-preview': True,
        }
        
        # Validation: Ensure the user's choice is valid
        if preferred_model not in self.models:
            raise ValueError(f"Invalid model '{preferred_model}'. "
                             f"Valid options: {list(self.models.keys())}")
            
        self.current_model = preferred_model

    def _get_next_available_model(self):
        """Sets the first model from the non-exhausted models as the currently selected model. 
        Raises error if no available model left."""
        for model_name, is_available in self.models.items():
            if is_available:
                print(f"🔄 Switching to fallback: {model_name}")
                self.current_model = model_name
                return 
    
        raise RuntimeError("❌ All available models are currently exhausted. "
                           "Please wait for quotas to reset.")

    def generate_content(self, contents, config=None):
        """
        Recursively attempts to generate content.
        If a model fails with a quota error, it marks it as unavailable 
        and retries with the next available model.
        """
        try:
            # Attempt generation
            response = self.client.models.generate_content(
                model=self.current_model,
                contents=contents,
                config=config
            )
            return response

        except Exception as e:
            # Check for Rate Limit / Quota errors
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"⚠️ Quota exceeded for {self.current_model}. Marking as unavailable...")
                
                # Update State: Mark current as failed
                self.models[self.current_model] = False
                
                # Switch to next available
                self._get_next_available_model()
            
                # Recursive Step: Try again with the new model
                return self.generate_content(contents, config)
            
            # If it's a logic error (e.g. invalid prompt), raise immediately
            raise e


# Free tier models have limits. In case we run out of a model quota, we'll 
# use one of the fallback models.
#
# Possible model names we can use are:
# - gemini-2.5-flash-lite
# - gemini-2.5-flash
# - gemini-3-flash-preview
try:
    model = GeminiModel(GEMINI_API_KEY, model_name='gemini-2.5-flash')
    print("✅ Connection Established. The Engine is ready.")
except Exception as e:
    print(f"❌ Error: {e}. Did you paste your API key?")

# 🏠 Homework: The Socratic Tutor

### The Scenario
You are building an educational bot for a history class.
The goal is NOT to give the student the answer.
The goal is to **guide** the student to the answer by asking questions (Socratic Method).

### The Task
Write a robust prompt that:
1.  **Persona:** Acts as Socrates (wise, patient, inquisitive).
2.  **Pattern:** Uses the **Flip/Interview Pattern** (never answers, only asks).
3.  **Constraint:** If the student gets it wrong, give a hint, not the answer.
4.  **Template:** At the end of the session, output a "Report Card" in a specific format.


In [ ]:
# @title (Hidden) Answer Key
socratic_prompt = """
ACT AS: Socrates, the wise philosopher.
OBJECTIVE: Help the student understand the cause of WWI.
RULES:
1. Never give the answer.
2. Ask probing questions to guide their thinking.
3. If they are stuck, provide a metaphorical hint.
4. Keep responses short (under 50 words).

FORMAT:
After 5 interactions, output a summary in this template:
--- REPORT CARD ---
Understanding Level: [Low/Med/High]
Key Concepts Missed: [List]
Next Study Topic: [Topic]
-------------------
"""